# 2 · Data prep — feature table and input file for the model

1. defines the **role of each column** (target / media / control / unused) and automatically verifies that duplicates are indeed duplicates;
2. creates calendar flags and **selects the holiday variant** based on the BIC;
3. validates the final matrix (NaNs, rank, VIF);
4. determines the hold-out set and the baseline scenario;
5. saves `data/processed/model_input.csv` and `config.json` for the `3_modeling` stage.

!!! Indicators that are **consequences of advertising** (sales, orders, customers) are not included in the model — this constitutes leakage.

In [21]:
import os, sys
from pathlib import Path
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# --- Colab ---
#PROJECT_DIR = "/content/drive/MyDrive/digis"
#  local ---
PROJECT_DIR = "."

if IN_COLAB:
    drive.mount("/content/drive")
root = Path(PROJECT_DIR).resolve()
if not (root / "notebooks").exists() and (root.parent / "notebooks").exists():
    root = root.parent
assert (root / "notebooks" / "common.py").exists(), (
    f"Not found {root}/notebooks/common.py.")
os.chdir(root / "notebooks"); sys.path.insert(0, str(root / "notebooks"))
print("root :", root, "| Colab:", IN_COLAB)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
root : /content/drive/MyDrive/digis | Colab: True


In [23]:
import json, numpy as np, pandas as pd, statsmodels.formula.api as smf, statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif
import importlib, common
importlib.reload(common)
from common import *
pd.set_option("display.width",160); pd.set_option("display.max_columns",30)

# ---- param
K_YEARLY     = 3
HOLDOUT_DAYS = 90
BASE_MONTH   = "2026-09"
df = load(); print(df.shape)

(730, 15)


In [24]:
### Automated audit: verify data integrity and logical constraints

assert (df.sales_inc / df.sales_ex).between(1 + TAX - 1e-3, 1 + TAX + 1e-3).all(), "sales_inc ≠ sales_ex × 1.2"
assert (df[CH].sum(axis=1) - df.spend_total).abs().max() < 0.01, "spend_total ≠ Σ channels"
assert ((df.bf + df.cm) <= df.window).all(), "BF/CM is not a subset of window"
assert (df.new <= df.buyers).all() and (df.orders >= df.buyers).all(), "Logical funnel constraint violated"

print("OK: sales_inc ≡ sales_ex · 1.2 | spend_total = Σ channels | bf,cm ⊂ window | new ≤ buyers ≤ orders")

leak = df[["sales_ex", "buyers", "orders"]].corrwith(df.new).round(3).to_dict()
print("Advertising consequences, correlation with target (new):", leak)

OK: sales_inc ≡ sales_ex · 1.2 | spend_total = Σ channels | bf,cm ⊂ window | new ≤ buyers ≤ orders
Advertising consequences, correlation with target (new): {'sales_ex': 0.98, 'buyers': 0.994, 'orders': 0.994}


## Calendar features

In [25]:
F = df[["date","new"]+CH+["promo"]].copy()
F["bf_cm_peak"]=bf_cm_peak(df)
md_=F.date.dt.strftime("%m-%d")
F["xmas_eve_day"]=md_.isin(["12-24","12-25"]).astype(int)
F["new_year"]=(md_=="01-01").astype(int)
F["holiday_low"]=(F.xmas_eve_day|F.new_year).astype(int)      # 24.12, 25.12, 01.01
dow=pd.get_dummies(F.date.dt.dayofweek,prefix="dow",drop_first=True).astype(int)
F=pd.concat([F,dow],axis=1)
F["t"]=np.arange(len(F))/len(F)                                # scaled trend 0..1
print(F[["bf_cm_peak","xmas_eve_day","new_year","holiday_low"]].sum().to_dict())

{'bf_cm_peak': 8, 'xmas_eve_day': 4, 'new_year': 2, 'holiday_low': 6}


## Holiday Flag Selection
Evaluated via BIC (OLS screening on `log(new)`, finalized in `3_modeling`):
- **A** — Christmas Eve & Day only (Dec 24–25);
- **B** — Combined low-activity flag (`holiday_low` for Dec 24, 25, and Jan 1);
- **C** — Two separate flags: `xmas_eve_day` + `new_year`.

**Selection rule:** Lowest BIC; if the difference is < 2, choose the simpler model.
*Result:* Variant C won with the lowest BIC (-843.06), showing distinct statistical significance for both Christmas and New Year drops.

In [26]:
S=F.copy(); S["y"]=np.log(S.new); S["doy"]=df.date.dt.dayofyear.values
Y=fourier(K_YEARLY,S.doy); S=pd.concat([S,Y],axis=1)
dcols=[c for c in S if c.startswith("dow_")]
base="y ~ "+"+".join(dcols+["bf_cm_peak","promo","t"]+list(Y.columns))
specs={"A: xmas_eve_day":["xmas_eve_day"],"B: holiday_low":["holiday_low"],"C: xmas_eve_day + new_year":["xmas_eve_day","new_year"]}
rows=[]
for n,cols in specs.items():
    m=smf.ols(base+"+"+"+".join(cols),S).fit(); rows.append((n,len(cols),m.rsquared,m.bic,m.aic))
cmp=pd.DataFrame(rows,columns=["варіант","прапорців","R2","BIC","AIC"]).round(2); print(cmp.to_string(index=False))
best=cmp.loc[cmp.BIC.idxmin()]; near=cmp[cmp.BIC<=best.BIC+2].sort_values("прапорців").iloc[0]
choice=near["варіант"]; HOLIDAY_COLS=specs[choice]
print("\nОбрано:",choice,"→",HOLIDAY_COLS)
m=smf.ols(base+"+"+"+".join(HOLIDAY_COLS),S).fit()
print(m.params[HOLIDAY_COLS].round(3).to_dict(), "| p:",m.pvalues[HOLIDAY_COLS].round(4).to_dict())

                   варіант  прапорців   R2     BIC     AIC
           A: xmas_eve_day          1 0.83 -840.22 -918.30
            B: holiday_low          1 0.82 -805.25 -883.33
C: xmas_eve_day + new_year          2 0.83 -843.06 -925.74

Обрано: C: xmas_eve_day + new_year → ['xmas_eve_day', 'new_year']
{'xmas_eve_day': -1.027, 'new_year': -0.279} | p: {'xmas_eve_day': 0.0, 'new_year': 0.0024}


## Final Feature Matrix Validation


In [27]:
CONTROLS = dcols + ["bf_cm_peak", "promo", "t"] + HOLIDAY_COLS
final = F[["date", "new"] + CH + CONTROLS].copy()

# Event candidate features for downstream hyperparameter tuning (evaluated in 3_tuning)
final["bf_window"] = df.window.values  # 8-day window: Monday before BF .. Cyber Monday
final["bf_day"] = df.bf.values        # Black Friday day
final["cm_day"] = df.cm.values        # Cyber Monday day

# Data integrity assertions
assert final.isna().sum().sum() == 0, "Matrix contains NaN values"
const = [c for c in CONTROLS if final[c].nunique() <= 1]
assert not const, f"Constant features detected: {const}"

# Matrix rank check
X = final[CH + CONTROLS].astype(float)
rank = np.linalg.matrix_rank(X.values)
print(f"Matrix rank: {rank} out of {X.shape[1]} → {'OK, full rank' if rank == X.shape[1] else 'RANK DEFICIENCY'}")

# Variance Inflation Factor (VIF) audit
Z = sm.add_constant(X)
v = {c: round(vif(Z.values, i), 1) for i, c in enumerate(Z.columns) if c != "const"}
print("VIF check:", {k: x for k, x in v.items() if x > 2.5} or "All VIF < 2.5")

print("Final dataset shape:", final.shape)
final.head()

Matrix rank: 15 out of 15 → OK, full rank
VIF check: All VIF < 2.5
Final dataset shape: (730, 20)


,date,new,bing,google,meta,pinterest,dow_1,dow_2,dow_3,dow_4,dow_5,dow_6,bf_cm_peak,promo,t,xmas_eve_day,new_year,bf_window,bf_day,cm_day
0,2024-10-01,458,334.72,2657.94,2393.94,512.70,1,0,0,0,0,0,0,0.0,0.000000,0,0,0,0,0
1,2024-10-02,468,310.00,3087.64,2462.26,468.30,0,1,0,0,0,0,0,0.0,0.001370,0,0,0,0,0
2,2024-10-03,476,321.55,3215.14,2965.07,469.93,0,0,1,0,0,0,0,0.0,0.002740,0,0,0,0,0
3,2024-10-04,504,391.43,3018.59,3110.63,512.49,0,0,0,1,0,0,0,0.0,0.004110,0,0,0,0,0
4,2024-10-05,425,379.63,2492.08,2738.47,550.13,0,0,0,0,1,0,0,0.0,0.005479,0,0,0,0,0


## Hold-out Set & Baseline Scenario

In [28]:
split_date = final.date.max() - pd.Timedelta(days=HOLDOUT_DAYS - 1)
print(
    f"train: {final.date.min().date()} → {(split_date - pd.Timedelta(days=1)).date()} "
    f"| hold-out: {split_date.date()} → {final.date.max().date()}"
)

base_m = final[final.date.dt.strftime("%Y-%m") == BASE_MONTH]
baseline = base_m[CH].mean().round(2).to_dict()
print(
    f"Baseline ({BASE_MONTH}), £/day: {baseline} "
    f"| Total: {round(sum(baseline.values()), 1)}"
)

ho = final[final.date >= split_date]
n_bf_cm = int(ho.bf_cm_peak.sum())
n_holidays = int(ho[HOLIDAY_COLS].sum().sum())
print(
    f"Hold-out BF/CM days: {n_bf_cm} | Holiday drop days: {n_holidays} "
    f"→ Event-free hold-out, evaluates baseline operations"
)

train: 2024-10-01 → 2026-07-02 | hold-out: 2026-07-03 → 2026-09-30
Baseline (2026-09), £/day: {'bing': 331.44, 'google': 2264.95, 'meta': 2509.4, 'pinterest': 634.51} | Total: 5740.3
Hold-out BF/CM days: 0 | Holiday drop days: 0 → Event-free hold-out, evaluates baseline operations


## Save

In [29]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
final.to_csv(OUT_DIR / "model_input.csv", index=False)
cfg = dict(
    target="new",
    channels=CH,
    controls=CONTROLS,
    k_yearly=K_YEARLY,
    holiday_choice=choice,
    holdout_start=str(split_date.date()),
    baseline_month=BASE_MONTH,
    baseline_daily_spend=baseline,
    target_cac=TARGET_CAC,
    optional_controls=OPTIONAL_CONTROLS,
)
json.dump(cfg, open(OUT_DIR / "config.json", "w"), indent=2, ensure_ascii=False)
print("Saved to", OUT_DIR)
print(json.dumps(cfg, indent=2, ensure_ascii=False))

Saved to /content/drive/MyDrive/digis/data/processed
{
  "target": "new",
  "channels": [
    "bing",
    "google",
    "meta",
    "pinterest"
  ],
  "controls": [
    "dow_1",
    "dow_2",
    "dow_3",
    "dow_4",
    "dow_5",
    "dow_6",
    "bf_cm_peak",
    "promo",
    "t",
    "xmas_eve_day",
    "new_year"
  ],
  "k_yearly": 3,
  "holiday_choice": "C: xmas_eve_day + new_year",
  "holdout_start": "2026-07-03",
  "baseline_month": "2026-09",
  "baseline_daily_spend": {
    "bing": 331.44,
    "google": 2264.95,
    "meta": 2509.4,
    "pinterest": 634.51
  },
  "target_cac": 40,
  "optional_controls": {
    "peak": [],
    "peak+window": [
      "bf_window"
    ],
    "peak+window+bf+cm": [
      "bf_window",
      "bf_day",
      "cm_day"
    ]
  }
}
